In [1]:
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image

import os
import cv2
import numpy as np
import pandas as pd
from collections import Counter
import tqdm


REF_DIR       = 'clean_references'
TEST_DIR      = 'test/test'
PRED_DIR      = 'test_pred'
SAMPLE_SUB    = 'sample_submission.csv'
OUTPUT_SUB    = 'submission_ip.csv'

MIN_AREA      = 10000
MORPH_KSIZE   = 9
HIST_CHANNELS = [0, 2]       # Hue & Value
HIST_BINS     = [50, 50]
HIST_RANGES   = [0, 180, 0, 256]
HIST_METHOD   = cv2.HISTCMP_CORREL

SIM_THRESH    = 0.7

In [ ]:
# Load a frozen ResNet-50 backbone (drop the final FC)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet = models.resnet50(pretrained=True).to(device).eval()
encoder = torch.nn.Sequential(*list(resnet.children())[:-1])  # B×2048×1×1

# precompute reference embeddings
tf_cnn = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

ref_embs = {}
#for fn in os.listdir(REF_DIR):
#    if not fn.lower().endswith(('.png','.jpg','.jpeg')):
#        continue
#    cls = os.path.splitext(fn)[0]
#    img = Image.open(os.path.join(REF_DIR,fn)).convert('RGB')
#    x   = tf_cnn(img).unsqueeze(0).to(device)                # 1×3×224×224
#    with torch.no_grad():
#        f = encoder(x).view(-1)                         # 2048
#        ref_embs[cls] = F.normalize(f, dim=0)           # L2 normalize

@torch.no_grad()
def embed_cnn(img_pil):
    x = tf_cnn(img_pil).unsqueeze(0).to(device)  # 1×3×224×224
    f = encoder(x).view(-1)                     # 2048
    return F.normalize(f, dim=0)                # L2 normalize


def load_reference_embs(ref_dir):
    ref_embs = {}
    for fn in sorted(os.listdir(ref_dir)):
        if not fn.lower().endswith(('.png','.jpg','.jpeg')):
            continue
        cls = os.path.splitext(fn)[0]
        img = Image.open(os.path.join(ref_dir,fn)).convert('RGB')
        ref_embs[cls] = embed_cnn(img)  # torch.Tensor(2048) on device
        print(f"Loaded CNN embedding for '{cls}'")
    return ref_embs

print("Loading reference embeddings…")
ref_embs = load_reference_embs(REF_DIR)
classes  = list(ref_embs.keys())


def classify_piece_by_cnn(patch_bgr, patch_mask, ref_embs):
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    f   = embed_cnn(pil)          
    best_cls, best_sim = None, -1.0
    for cls, r_emb in ref_embs.items():
        sim = float(torch.dot(f, r_emb))  # both on device
        if sim > best_sim:
            best_sim, best_cls = sim, cls
    return best_cls, best_sim

In [ ]:
def segment_pieces(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    border = np.hstack([gray[0,:], gray[-1,:], gray[:,0], gray[:,-1]])
    bg_val = int(np.median(border))
    diff = cv2.absdiff(gray, np.full_like(gray, bg_val))
    _, mask = cv2.threshold(diff, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kern = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (MORPH_KSIZE, MORPH_KSIZE)
    )
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kern, iterations=3)
    mask = cv2.dilate(mask, kern, iterations=2)
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    pieces = []
    for c in contours:
        if cv2.contourArea(c) < MIN_AREA:
            continue
        x,y,w_box,h_box = cv2.boundingRect(c)
        piece_mask = np.zeros_like(gray)
        cv2.drawContours(piece_mask, [c], -1, 255, thickness=-1)
        pieces.append((piece_mask, (x,y,w_box,h_box)))
    return pieces

In [ ]:
col_to_ref = {
    "Jelly White":        "Jelly_White",
    "Jelly Milk":         "Jelly_Milk",
    "Jelly Black":        "Jelly_Black",
    "Amandina":           "Amandina",
    "Crème brulée":       "Creme_brulee",
    "Triangolo":          "Triangolo",
    "Tentation noir":     "Tentation_noir",
    "Comtesse":           "Comtesse",
    "Noblesse":           "Noblesse",
    "Noir authentique":
    "Noir_authentique",
    "Passion au lait":    "Passion_au_lait",
    "Arabia":             "Arabia",
    "Stracciatella":      "Stracciatella",
}

In [ ]:
def predict_and_draw(img_bgr, ref_embs):
    """
    Returns:
      counts: dict[class_name->count],
      vis: BGR image with boxes+labels drawn
    """
    pieces = segment_pieces(img_bgr)
    cnts   = Counter()
    vis    = img_bgr.copy()
    for mask, (x,y,w_box,h_box) in pieces:
        patch   = img_bgr[y:y+h_box, x:x+w_box]
        submask = mask[y:y+h_box, x:x+w_box]
        cls, score = classify_piece_by_cnn(patch, submask, ref_embs)
        if score >= SIM_THRESH:
            cnts[cls] += 1
            # draw
            cv2.rectangle(vis, (x,y), (x+w_box, y+h_box), (255,0,255), 3)
            text = f"{cls}:{score:.2f}"
            cv2.putText(vis, text, (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 3, (255,0,255), 2,
                        lineType=cv2.LINE_AA)
    # fill zeros for missing
    for c in classes:
        cnts.setdefault(c, 0)
    return dict(cnts), vis

os.makedirs(PRED_DIR, exist_ok=True)
sample_df = pd.read_csv(SAMPLE_SUB)
output_df = sample_df.copy()

# sanity check mapping
for c in sample_df.columns:
    if c!="id" and c not in col_to_ref:
        raise ValueError(f"No mapping for column '{c}'")

for idx, row in tqdm.tqdm(sample_df.iterrows(), total=len(sample_df)):
    img_id = str(row["id"])
    fn     = f"L{img_id}.jpg"
    path   = os.path.join(TEST_DIR, fn)
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        path = os.path.join(TEST_DIR, f"L{img_id}.JPG")
        img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot load {fn} in {TEST_DIR}")

    # predict + draw
    pred_counts, vis = predict_and_draw(img_bgr, ref_embs)

    # fill submission counts using the mapping
    for sub_col, ref_key in col_to_ref.items():
        output_df.at[idx, sub_col] = int(pred_counts.get(ref_key, 0))

    # save visualization
    out_vis = os.path.join(PRED_DIR, fn)
    cv2.imwrite(out_vis, vis)

print("Writing submission.csv…")
output_df.to_csv(OUTPUT_SUB, index=False)
print("Saved detections to folder:", PRED_DIR)
print("Done.")